# Programa Agenda de Contatos

### Descrição do Projeto
Este exercício consiste no desenvolvimento de uma agenda de contatos completa, focada em praticidade e robustez. O sistema permite as operações de: **Adicionar**, **Editar**, **Pesquisar**, **Listar** e **Apagar** contatos, além de um controle seguro de saída.

### Destaques Técnicos Implementados:
*   **Estrutura de Dados**: Organização eficiente utilizando dicionários para armazenar Nome, Telefone e E-mail.
*   **Validação Inteligente**: Uso de expressões regulares (`re`) para garantir que telefones (DDD + 9 dígitos) e e-mails estejam no formato correto.
*   **Interface Amigável**: Uso de cores ANSI para destacar mensagens de sucesso, erro e títulos, além de tratamento de exceções para evitar travamentos por digitação incorreta.
*   **Controle de Fluxo**: Sistema de confirmação de exclusão e saída, garantindo a segurança dos dados durante o uso.


## 1. Importações e Utilidades

### Importações
Este bloco reúne todas as bibliotecas necessárias para o funcionamento da agenda, incluindo manipulação de expressões regulares, interação com o sistema operacional e controle de execução.


In [119]:
import re # Manipulação de expressões regulares para validação de padrões
import os # Interface com o sistema operacional para comandos de terminal
import sys # Interação com o interpretador e ambiente de execução
import json # Manipulação de arquivos JSON para persistência de dados

# As bibliotecas acima garantem a portabilidade e a integridade dos dados na agenda.

### Configuração de Estilos e Feedback Visual
Nesta etapa, estabelecemos a identidade visual do programa. São definidas as constantes de cores ANSI e funções auxiliares que garantem consistência na interface, permitindo diferenciar títulos, mensagens de sucesso, alertas e erros de forma clara e padronizada.


#### Definição de Códigos ANSI
Este bloco define as constantes ANSI responsáveis por aplicar cores e estilos no terminal. Elas serão utilizadas pelas funções de exibição para destacar mensagens e melhorar a experiência visual do usuário.


In [120]:
# Configuração de estilos visuais para o terminal utilizando sequências de escape ANSI
COR_RESET = '\033[0m'
COR_TITULO = '\033[1;36m'  # Ciano Negrito
COR_SUCESSO = '\033[1;32m' # Verde Negrito
COR_ERRO = '\033[1;31m'    # Vermelho Negrito
COR_ALERTA = '\033[1;33m'   # Amarelo Negrito

#### Função de Exibição de Mensagens
A função `exibir_mensagem` aplica os estilos visuais definidos anteriormente para formatar e exibir mensagens coloridas no terminal.  
Ela centraliza a lógica de saída, garantindo consistência em todo o programa.


In [121]:
def exibir_mensagem(texto, tipo='sucesso'):
    """
    Formata e exibe mensagens coloridas no terminal.

    Args:
        texto (str): O conteúdo da mensagem a ser exibida.
        tipo (str): A categoria do estilo ('titulo', 'sucesso', 'erro', 'alerta').

    Returns:
        None
    """
    estilos = {
        'titulo': COR_TITULO,
        'sucesso': COR_SUCESSO,
        'erro': COR_ERRO,
        'alerta': COR_ALERTA
    }
    cor = estilos.get(tipo, COR_RESET)
    print(f"{cor}{texto}{COR_RESET}")

## 2. Infraestrutura de Validação e Fluxo de Entrada
Esta seção reúne as funções que garantem a integridade dos dados da agenda.  
Aqui estão incluídas:  
* **Validação de Formatos**: uso de expressões regulares para garantir que telefones e e-mails sigam padrões corretos.  
* **Gestão de Fluxo**: exceção personalizada `OperacaoCancelada` para permitir cancelamentos seguros.  
* **Busca e Coleta**: funções utilitárias para localizar contatos e coletar múltiplos dados sem duplicação de código.


### Definição do Arquivo de Persistência
Este bloco estabelece o nome do arquivo JSON que será utilizado para salvar e recuperar os contatos da agenda.  
Ele funciona como ponto central de armazenamento, garantindo que os dados permaneçam disponíveis entre diferentes execuções do programa.


In [122]:
# Define o nome do arquivo JSON onde os contatos serão armazenados permanentemente
ARQUIVO_AGENDA = 'agenda_contatos.json'

### Função de Carregamento da Agenda
Esta função lê o arquivo JSON definido em `ARQUIVO_AGENDA` e retorna os contatos armazenados.  
Se o arquivo não existir ou estiver corrompido, a agenda é inicializada como vazia, garantindo que o programa continue funcionando de forma segura.

#### Inicialização Global da Agenda
Após definir as funções de persistência, a agenda é carregada automaticamente a partir do arquivo.  
Isso assegura que os contatos previamente salvos estejam disponíveis desde o início da execução do programa.


In [123]:
def carregar_agenda():
    """
    Carrega a agenda de contatos a partir do arquivo JSON.
    Se o arquivo não existir ou estiver corrompido, retorna um dicionário vazio.
    """
    try:
        with open(ARQUIVO_AGENDA, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        exibir_mensagem("Arquivo de agenda não encontrado. Iniciando nova agenda.", 'alerta')
        return {}
    except json.JSONDecodeError:
        exibir_mensagem("Erro ao ler arquivo: formato inválido. Iniciando nova agenda.", 'erro')
        return {}
    except Exception as e:
        exibir_mensagem(f"Erro inesperado ao carregar: {e}", 'erro')
        return {}

# Inicialização automática da agenda global
agenda = carregar_agenda()

### Função de Salvamento da Agenda
Esta função grava o conteúdo atual da agenda no arquivo JSON definido em `ARQUIVO_AGENDA`.  
Ela garante que qualquer alteração realizada (adição, edição ou exclusão de contatos) seja registrada de forma permanente.  
Em caso de sucesso, uma mensagem de confirmação é exibida; se ocorrer erro, o usuário é informado para manter a transparência do processo.


In [124]:
def salvar_agenda():
    """
    Salva o dicionário da agenda no arquivo JSON definido em ARQUIVO_AGENDA.
    Exibe uma mensagem de sucesso ou erro dependendo do resultado.
    """
    try:
        with open(ARQUIVO_AGENDA, 'w', encoding='utf-8') as f:
            # Salva com indentação para melhor leitura humana
            json.dump(agenda, f, indent=4, ensure_ascii=False)
        exibir_mensagem("Agenda salva com sucesso!", 'sucesso')
    except Exception as e:
        exibir_mensagem(f"Erro ao salvar a agenda: {e}", 'erro')

### Exceção de Cancelamento
A classe `OperacaoCancelada` define uma exceção específica para sinalizar quando o usuário decide interromper uma operação digitando 'sair'.
Isso permite que o fluxo do programa seja controlado de forma elegante e previsível.

In [125]:
class OperacaoCancelada(Exception):
    """Exceção personalizada para sinalizar cancelamento pelo usuário."""
    pass

### Função de Validação de Formatos
A função `validar_formato` garante que os dados inseridos (telefone ou e-mail) estejam no padrão correto, utilizando expressões regulares.

In [126]:
def validar_formato(valor, tipo):
    """
    Valida se o formato do dado inserido está logicamente correto.
    """
    if tipo == 'telefone':
        return bool(re.match(r'^\d{11}$', valor))
    elif tipo == 'email':
        return bool(re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', valor))
    return False

### Função de Entrada Validada
A função `validar_input` gerencia a interação com o usuário, solicitando dados e validando-os.
Caso o usuário digite 'sair', a operação é interrompida pela exceção `OperacaoCancelada`.


In [127]:
def validar_input(prompt, tipo):
    """
    Gerencia a interface de entrada e lança OperacaoCancelada se o usuário digitar 'sair'.
    """
    while True:
        entrada = input(prompt).strip()

        if entrada.lower() == 'sair':
            raise OperacaoCancelada()

        if validar_formato(entrada, tipo):
            return entrada
        else:
            mensagem = "O telefone deve conter 11 dígitos numéricos." if tipo == 'telefone' else "Insira um endereço de e-mail válido."
            exibir_mensagem(f"Erro: {mensagem}", 'erro')

### Função de Busca de Contato
A função `buscar_contato_por_nome` localiza um contato na agenda de forma insensível a maiúsculas, retornando a chave original se encontrado.


In [128]:
def buscar_contato_por_nome(nome_pesquisa):
    """
    Localiza a chave original do contato no dicionário de forma insensível a maiúsculas.
    """
    for nome_agenda in agenda.keys():
        if nome_agenda.lower() == nome_pesquisa.lower():
            return nome_agenda
    return None

### Função de Coleta de Dados de Contato
A função `obter_dados_contato` centraliza a coleta de telefone e e-mail, aplicando validação e permitindo cancelamento seguro.


In [129]:
def obter_dados_contato(ignorar_email=False, ignorar_tel=False):
    """
    Coleta e valida múltiplos campos. Lança OperacaoCancelada se necessário.
    """
    dados = {}

    if not ignorar_tel:
        dados['telefone'] = validar_input("\nTelefone (11 dígitos ou 'sair' para cancelar): ", 'telefone')

    if not ignorar_email:
        dados['email'] = validar_input("\nE-mail ('sair' para cancelar): ", 'email')

    return dados

### Função de Limpeza de Tela Multiplataforma
Esta função mantém a interface organizada, limpando o console após cada operação.  
Ela detecta se o programa está sendo executado no Google Colab ou em um terminal local (Windows/Linux) e aplica o comando adequado para evitar conflitos e garantir estabilidade.


In [130]:
def limpar_tela():
    """
    Gerencia a limpeza do console de forma multiplataforma.

    Detecta se a execução ocorre no Google Colab para evitar conflitos de entrada
    e utiliza comandos de sistema para Windows ou Unix.

    Returns:
        None
    """
    is_colab = 'google.colab' in sys.modules

    try:
        if not is_colab:
            os.system('cls' if os.name == 'nt' else 'clear')
    except:
        pass

###Função Pedir Nome
Esse código solicita ao usuário um nome que será consultado na agenda.

In [131]:
def pedir_nome():
    """
    Exibe um prompt para o usuário e captura o nome para busca na agenda.

    Returns:
        str: O nome digitado pelo usuário, sem espaços em branco nas extremidades.
    """
    print("\n===== DIGITE O NOME DO CONTATO =====\n")
    p_nome = input("Nome: ").strip().title()
    return p_nome

##3. Operações Principais da Agenda

###Menu Principal
Esse Procedimento apresenta na tela as opções que podem ser usadas na agenda.


In [132]:
def menu():
    """
    Exibe o menu principal com as opções de navegação do sistema.

    Returns:
        None
    """
    print(f"\n{COR_TITULO}===== AGENDA DE CONTATOS ====={COR_RESET}")
    print("""
    1 - Adicionar novo contato
    2 - Editar contato
    3 - Pesquisar contato
    4 - Listar contatos
    5 - Apagar um contato
    6 - Sair
    """)

###Função Adicionar um novo contato
Essa função será usada para criar um novo contato que será inserido na agenda.

In [133]:
def adicionar():
    """
    Solicita dados do usuário, armazena um novo contato na agenda e salva no arquivo.
    """
    try:
        exibir_mensagem("\n===== ADICIONAR NOVO CONTATO =====", 'titulo')
        nome = input("\nNome Completo: ").strip().title()

        if not nome:
            exibir_mensagem("Erro: O nome não pode ser vazio.", 'erro')
            return

        if buscar_contato_por_nome(nome):
            exibir_mensagem("Aviso: Já existe um contato com este nome.", 'alerta')
            return

        # Coleta e persiste os novos dados
        agenda[nome] = obter_dados_contato()
        salvar_agenda()

        exibir_mensagem(f"\nContato '{nome}' adicionado com sucesso!", 'sucesso')

    except OperacaoCancelada:
        exibir_mensagem("Operação de adicionar contato cancelada.", 'alerta')


###Procedimento Pesquisar Contato

In [134]:
def pesquisar():
    """
    Busca um contato pela chave do dicionário de forma insensível a maiúsculas.

    Returns:
        str: O nome (chave) do contato se encontrado, ou None caso contrário.
    """
    nome_digitado = pedir_nome()
    nome_encontrado = buscar_contato_por_nome(nome_digitado)

    if nome_encontrado:
        contato = agenda[nome_encontrado]
        exibir_mensagem("\n===== CONTATO ENCONTRADO =====", 'titulo')
        print(f"Nome: {nome_encontrado.title()}\nTelefone: {contato['telefone']}\nE-mail: {contato['email']}")
        print("\n----------------------------------------------")
        return nome_encontrado

    exibir_mensagem(f"\nContato '{nome_digitado}' não encontrado!", 'erro')
    return None

### Função Editar Contato
Esta função localiza um contato existente e permite atualizar telefone, e-mail ou ambos os campos.  
O usuário pode escolher a opção desejada ou cancelar a operação de forma segura.


In [135]:
def editar():
    """
    Localiza um contato e permite a atualização de seus campos via interface interativa.

    Returns:
        None
    """
    nome_contato = pesquisar()
    if nome_contato:
        try:
            exibir_mensagem(f"\n===== EDITAR CONTATO: {nome_contato.title()} =====", 'titulo')
            print("O que você deseja alterar?\n1 - Telefone\n2 - E-mail\n3 - Ambos\n4 - Cancelar")

            opcao = input("\nEscolha uma opção: ")

            if opcao == '1':
                agenda[nome_contato].update(obter_dados_contato(ignorar_email=True))
            elif opcao == '2':
                agenda[nome_contato].update(obter_dados_contato(ignorar_tel=True))
            elif opcao == '3':
                agenda[nome_contato].update(obter_dados_contato())
            elif opcao == '4':
                raise OperacaoCancelada()
            else:
                exibir_mensagem("Opção inválida!", 'erro')
                return

            salvar_agenda()

            exibir_mensagem("\nDados atualizados com sucesso!", 'sucesso')

        except OperacaoCancelada:
            exibir_mensagem("Edição cancelada.", 'alerta')

### Função Apagar Contato
Esta função remove um contato da agenda após confirmação explícita do usuário.  
O processo garante segurança contra exclusões acidentais, oferecendo a opção de cancelar antes da remoção definitiva.


In [136]:
def apagar():
    """
    Remove um contato do dicionário após confirmação explícita do usuário.

    Returns:
        None
    """
    nome_contato = pesquisar()

    if nome_contato:
        exibir_mensagem(f"\nDeseja realmente apagar o contato '{nome_contato}'?", 'alerta')
        decisao = input("Digite S para Sim e N para Não: ").strip().upper()

        if decisao == "S":
            del agenda[nome_contato]

            salvar_agenda()

            exibir_mensagem("\nContato removido com sucesso!", 'sucesso')
        else:
            exibir_mensagem("\nOperação cancelada.", 'alerta')

### Função de Listagem com Contador Dinâmico
A função `listar` é responsável pela exibição organizada de todos os registros armazenados na agenda. Para auxiliar no gerenciamento, o sistema realiza o cálculo automático da quantidade de contatos existentes, apresentando um contador dinâmico no cabeçalho da lista que reflete o volume total de dados em tempo real.

In [137]:
def listar():
    """
    Exibe todos os contatos armazenados no dicionário de forma tabular e colorida.

    Returns:
        None
    """
    total_contatos = len(agenda)

    if total_contatos == 0:
        exibir_mensagem("\n===== AGENDA VAZIA =====", 'erro')
        return

    exibir_mensagem(f"\n===== LISTA DE CONTATOS (Total: {total_contatos}) =====", 'titulo')

    for nome, dados in agenda.items():
        print(f"Nome: {nome.title()}\nTelefone: {dados['telefone']}\nE-mail: {dados['email']}")
        print("---------------------------------------------------")

## 4. Execução do Programa (Loop Principal)


### Controle de Fluxo e Encerramento Seguro
Esta seção mantém o programa em execução contínua, exibindo o menu e direcionando para as funções principais.  
Inclui mecanismos de segurança, como a confirmação antes de encerrar a aplicação, para evitar saídas acidentais.


#### Função de Processamento de Opções
A função `processar_opcao` interpreta a escolha do usuário no menu e encaminha o fluxo para a operação correspondente.  
Retorna `True` para continuar a execução ou `False` para encerrar o programa.


In [138]:
def processar_opcao(opcao):
    """
    Encaminha o fluxo do programa para a operação escolhida no menu.

    Args:
        opcao (int): O número da opção selecionada pelo usuário.

    Returns:
        bool: True para manter o programa em execução, False para encerrar.
    """
    if opcao == 1:
        adicionar()
    elif opcao == 2:
        editar()
    elif opcao == 3:
        pesquisar()
    elif opcao == 4:
        listar()
    elif opcao == 5:
        apagar()
    elif opcao == 6:
        confirmacao = input("\nDeseja realmente sair? (S/N): ").strip().upper()
        if confirmacao == 'S':
            salvar_agenda()
            exibir_mensagem("Finalizando programa...", 'alerta')
            return False
    else:
        exibir_mensagem("Opção inválida! Escolha entre 1 e 6.", 'erro')

    return True

#### Loop Principal da Agenda
A função `executar_agenda` mantém o programa em execução contínua, exibindo o menu, capturando entradas e tratando exceções globais.  
É o coração da aplicação, garantindo que todas as operações funcionem de forma integrada e segura.


In [139]:
def executar_agenda():
    """
    Loop principal que sustenta o ciclo de vida da aplicação.

    Gerencia a exibição do menu, captura de opções e tratamento de exceções globais.
    """
    while True:
        menu()

        try:
            entrada = input("Digite a opção desejada (1-6): ").strip()

            if not entrada:
                continue

            if not entrada.isdigit():
                exibir_mensagem("\nErro: Por favor, utilize apenas números.", 'erro')
                input("Pressione Enter...")
                limpar_tela()
                continue

            if not processar_opcao(int(entrada)):
                break

            input("\nPressione Enter para continuar...")
            limpar_tela()

        except KeyboardInterrupt:
            print("\n")
            exibir_mensagem("Execução interrompida pelo usuário.", 'alerta')
            break
        except Exception as e:
            exibir_mensagem(f"\nOcorreu um erro inesperado: {e}", 'erro')
            input("Pressione Enter...")
            limpar_tela()

In [140]:
executar_agenda()


===== AGENDA DE CONTATOS =====

    1 - Adicionar novo contato
    2 - Editar contato
    3 - Pesquisar contato
    4 - Listar contatos
    5 - Apagar um contato
    6 - Sair
    
Digite a opção desejada (1-6): 6

Deseja realmente sair? (S/N): s
Agenda salva com sucesso!
Finalizando programa...


#Bloco de Teste Manual e Verificação de Persistência
Este bloco é utilizado apenas para validar se a função de persistência está funcionando corretamente.  
Ele cria um contato fictício de teste, salva a agenda e verifica se o arquivo JSON foi gerado no diretório do Colab.  
Além disso, exibe instruções para que o usuário confirme manualmente a recuperação dos dados após reiniciar o ambiente de execução.


In [141]:
# Bloco de Teste Manual e Verificação de Persistência
print("--- Testando Persistência ---")
agenda['Teste Persistência'] = {'telefone': '11999999999', 'email': 'teste@email.com'}
salvar_agenda()

# Verificando se o arquivo físico existe no diretório do Colab
if os.path.exists(ARQUIVO_AGENDA):
    exibir_mensagem(f"Confirmação: O arquivo '{ARQUIVO_AGENDA}' foi criado com sucesso!", 'sucesso')
    with open(ARQUIVO_AGENDA, 'r') as f:
        print("Conteúdo atual do arquivo:")
        print(f.read())
else:
    exibir_mensagem("Erro: O arquivo de persistência não foi localizado.", 'erro')

print("\n--- Instrução de Verificação Manual ---")
print("1. Use o menu para adicionar um contato real.")
print("2. Vá em 'Ambiente de execução' -> 'Reiniciar ambiente de execução'.")
print("3. Execute as células de definição novamente.")
print("4. Chame listar() ou pesquisar() para confirmar que os dados voltaram.")

--- Testando Persistência ---
Agenda salva com sucesso!
Confirmação: O arquivo 'agenda_contatos.json' foi criado com sucesso!
Conteúdo atual do arquivo:
{
    "Teste": {
        "telefone": "11123456789",
        "email": "teste@teste.com.br"
    },
    "Teste Persistência": {
        "telefone": "11999999999",
        "email": "teste@email.com"
    },
    "Carlos": {
        "telefone": "11913264859",
        "email": "carlos@teste.com"
    }
}

--- Instrução de Verificação Manual ---
1. Use o menu para adicionar um contato real.
2. Vá em 'Ambiente de execução' -> 'Reiniciar ambiente de execução'.
3. Execute as células de definição novamente.
4. Chame listar() ou pesquisar() para confirmar que os dados voltaram.
